SOC -> ISCO -> PSOC

In [1]:
import pandas as pd

In [41]:
# Open the file
filepath = '../data/lfs_parquet/parquet/LFS PUF January 2025.parquet'
lf_df = pd.read_parquet(filepath)

# Get only from the labor force and ony those who have a PSOC Code
is_lf = (lf_df['PUFNEWEMPSTAT'] == '1') | (lf_df['PUFNEWEMPSTAT'] == '2')
lf_df = lf_df[is_lf].copy()

lf_code_df = lf_df.dropna(subset=['PUFC14_PROCC']).copy()

In [54]:
# Open the files for aioe and complementarity
filepath = '../data/psoc_aioe.csv'
psoc_aioe_df = pd.read_csv(filepath, index_col=1).drop(columns=['Unnamed: 0'])['AIOE'].copy()

filepath = '../data/psoc_comple.csv'
psoc_comple_df = pd.read_csv(filepath, index_col=0).mean(axis=1).copy()

# Concatenate the two indices
ai_df = pd.concat([psoc_aioe_df, psoc_comple_df], axis=1).copy()
ai_df.columns = ['AIOE', 'Complementarity']
ai_df.loc[:, 'Code'] = ai_df.index
code_ai_df = ai_df.groupby('Code')[['AIOE', 'Complementarity']].mean()

# Get the mapping and dictionary
aioe_map = dict(zip(code_ai_df.index, code_ai_df.AIOE))
complementarity_map = dict(zip(code_ai_df.index, code_ai_df.Complementarity))

In [43]:
filepath = '../data/psoc_soc.csv'
psoc_soc_df = pd.read_csv(filepath)
psoc_soc_df.columns = ['PSOC', 'SOC']
psoc_soc_df['PSOC_abbreviated'] = psoc_soc_df['PSOC'].astype(str).str[:2]

In [44]:
soc_most_common = psoc_soc_df.groupby('PSOC_abbreviated')['SOC'].agg(lambda x : x.mode().iloc[0])
psoc_abbreviated_soc = soc_most_common.to_dict()

In [45]:
lf_code_df.loc[:, 'SOC'] = lf_code_df['PUFC14_PROCC'].map(psoc_abbreviated_soc)

In [80]:
lf_code_df['SOC'].str.strip().map(aioe_map)

0         0.230704
1         1.369843
5         1.429095
8        -1.153153
11             NaN
            ...   
173270         NaN
173275         NaN
173276         NaN
173279   -1.153153
173281   -1.153153
Name: SOC, Length: 77077, dtype: float64

In [66]:
lf_code_df.SOC.isnull().sum()

np.int64(188)